In [ ]:
from dotenv import load_dotenv
load_dotenv()

from typing_extensions import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver  #x SHORTTERMMEMORY

from langgraph.types import Command, interrupt  #human-in-the-loop orchestration.  interrupt() ferma il graph e lo fa aspettare per input umano 

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode  #x node executor per tool
import os
import re  #x regex engine, per pattern matching

# Configuration
BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen3"
llm = ChatOllama(
    model=MODEL_NAME, 
    base_url=BASE_URL)

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

@tool
def transfer_money(amount:int, recipient:str):
    """
    Transfer money. Large transfers require approval.
    Args:
        amount: Amount in dollars
        recipient: Recipient name
    """
    if amount > 1000:  #
        approval = interrupt(  #richiesta approvazione umana
            {
                "type": "approval_required",
                "amount": amount,
                "recipient": recipient
            }
        )
        #quando il graph riparte, approval conterrà la decisione umana
        if approval.get("decision") != "approve":
            return "Transfer is cancelled!"
    return f"Transferred {amount} to {recipient}"

transfer_money.invoke({'amount': 100, 'recipient': 'Laxmi Kant'})

In [ ]:
#dizionario regex patterns
patterns = {
        "SSN": r'\b\d{3}-\d{2}-\d{4}\b',  #SSN: 123-45-6789
        "Credit Card": r'\b\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}\b',  #Credit Card: 1234-5678-9012-3456
        "Mobile Number": r'\b(\+?\d{1,3}[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b',  #Mobile: +1-234-567-8900
        "Email": r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',  #Email: user@example.com
        "URL/Link": r'https?://[^\s]+|www\.[^\s]+'  #URL: http://example.com or www.example.com
    }

def guardrail_node(state:AgentState):  #safety middleware node
    last_message = state['messages'][-1].content
    for pii_type, pattern in patterns.items():
        if re.search(pattern, last_message):  #cerca match pattern-mex
            return {
                "messages": [SystemMessage(f"Request Blocked: Contains {pii_type}. Please don't share sensitive personal information")]
            }
    return {"messages": []}   #update stato shared, in questo caso resetto

In [ ]:
def agent_node(state:AgentState):
    tools = [transfer_money]
    llm_with_tools = llm.bind_tools(tools)
    system_message = SystemMessage("You are a financial assistant. use transfer_money tool to transfer the money.")
    messages = [system_message] + state['messages']  #concatena 
    response = llm_with_tools.invoke(messages)
    if hasattr(response, 'tool_calls') and response.tool_calls:
        for tc in response.tool_calls:
            print(f"[AGENT] called Tool {tc.get('name', '?')} with args {tc.get('args', '?')}")
    else:
        print(f"[AGENT] Responding...")
    return {"messages": [response]}

In [ ]:
def should_continue(state: AgentState):
    last = state['messages'][-1]
    if hasattr(last, 'tool_calls') and last.tool_calls:
        return "tools"
    else:
        return END
    
def guardrail_router(state:AgentState):
    last = state['messages'][-1]
    if isinstance(last, SystemMessage):  #ISINSTANCE per verificare se è di tipo SystemMessage
        last.pretty_print()
        return  END
    return 'agent'
    

In [ ]:
import os  #x fiesystem operations

db_name = "db/checkpoints.db"

# if os.path.exists("db_name"):
#     os.removedirs(db_name)

import sqlite3  #x sqlite backend per checkpointing
os.makedirs('db', exist_ok=True)  #cre folder 'db', se esiste gia allora non da errore

In [ ]:
def create_agent():
    builder = StateGraph(AgentState)
    # add all nodes
    builder.add_node("guardrail", guardrail_node)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode([transfer_money]))
    # add egdes
    builder.add_edge(START, "guardrail")
    builder.add_conditional_edges("guardrail", guardrail_router, ['agent', END])
    builder.add_conditional_edges("agent", should_continue, ['tools', END])
    builder.add_edge("tools", "agent")
    #fresh connection
    conn = sqlite3.connect(
        db_name, 
        check_same_thread=False)
    checkpointer = SqliteSaver(conn)
    graph = builder.compile(checkpointer=checkpointer)
    return graph

agent = create_agent()
agent

In [ ]:
config = {"configurable": {'thread_id': 'demo-2'}}

query = "Transfer 500 to laxmi Kant."
result = agent.invoke({
    "messages": [HumanMessage(query)]},
    config)
result

query = "Transfer 5000 to laxmi Kant."
result = agent.invoke({
    "messages": [HumanMessage(query)]}, 
    config)
result

In [ ]:

if '__interrupt__' in result:  #check 'graph fermato?'
    interrupt_data = result['__interrupt__'][0]  #recupera payload interrupt (che solitamente invece andrebbe al frontend)
    result = agent.invoke(  #here riprendi il graph in sospeso
        Command(resume={'decision': input("do you approve or reject?")}), #Command fa capire a langgraph che non è un nuovo input, ma la continuazione del graph, dopo che l'utente INSERISCE L'INPUT (approve o reject), quindi poi langgraph fa state+={'decision': 'approve'}
        config
    )
    result['messages'][-1].pretty_print()

#e.g.se graph si è fermato allora in result hai 
# {
#   "__interrupt__": [
#      {
#         "type": "approval_required",
#         "amount": 5000
#      }
#   ]
# }

In [ ]:

def chat(query, thread_id):
    config = {"configurable": {'thread_id': thread_id}}
    result = agent.invoke({
        "messages": [HumanMessage(query)]}, 
        config)
    if '__interrupt__' in result:
        # use this data in front end design for approval workflow
        interrupt_data = result['__interrupt__'][0]
        result = agent.invoke(  #here riprendi il graph in sospeso
            Command(resume={'decision': input("do you approve or reject?")}),
            config
        )
    result['messages'][-1].pretty_print()

In [ ]:
thread_id = "demo-3"
query = "Transfer 5000 to laxmi Kant."
chat(query, thread_id)

thread_id = "demo-3"
query = "Transfer 5000 to udemy@kgptalkie.com."
chat(query, thread_id)

thread_id = "demo-3"
query = "Transfer 5000 to card 1234-5678-9012-3456."
chat(query, thread_id)

thread_id = "demo-3"
query = "Transfer 5000 to laxmi kant and send receipt to https://example.com/receipt."
chat(query, thread_id)

thread_id = "demo-3"
query = "Transfer 5000 to laxmi kant and send receipt to +1-234-567-8900."
chat(query, thread_id)